In [1]:
# ============================================================
# GOLD INSIGHT INDIA
# Interactive Gold Price Analysis Dashboard
# Google Colab | Pandas | Matplotlib | Seaborn | ipywidgets
# ============================================================


# 1. IMPORT LIBRARIES

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets

from IPython.display import display, clear_output
from math import ceil


# ============================================================
# 2. LOAD DATASET
# ============================================================

DATA_URL = "https://datahub.io/core/gold-prices/r/monthly.csv"

print("Loading gold price dataset...")

data = pd.read_csv(DATA_URL)

data.columns = [col.strip().lower() for col in data.columns]

date_col = next(
    (c for c in data.columns if "date" in c),
    None
)

price_col = next(
    (
        c for c in data.columns
        if "price" in c or "value" in c
    ),
    None
)

if date_col is None or price_col is None:
    raise ValueError(
        f"Could not identify date and price columns: {list(data.columns)}"
    )

data = data[[date_col, price_col]].copy()

data.columns = ["Date", "Gold_USD_per_Ounce"]

data["Date"] = pd.to_datetime(
    data["Date"], errors="coerce"
)

data["Gold_USD_per_Ounce"] = pd.to_numeric(
    data["Gold_USD_per_Ounce"], errors="coerce"
)

data = data.dropna()

data = data.sort_values("Date")

data = data.drop_duplicates(
    subset=["Date"], keep="last"
).reset_index(drop=True)

if data.empty:
    raise ValueError("No valid data was loaded.")


# ============================================================
# 3. PREPARE DATA
# ============================================================

data["Year"] = data["Date"].dt.year

data["Month"] = data["Date"].dt.month

data["Month_Name"] = data["Date"].dt.strftime("%b")

data["Monthly_Return"] = (
    data["Gold_USD_per_Ounce"].pct_change() * 100
)

data["MA_3"] = (
    data["Gold_USD_per_Ounce"].rolling(3).mean()
)

data["MA_6"] = (
    data["Gold_USD_per_Ounce"].rolling(6).mean()
)

data["MA_12"] = (
    data["Gold_USD_per_Ounce"].rolling(12).mean()
)

data["MA_24"] = (
    data["Gold_USD_per_Ounce"].rolling(24).mean()
)

data["Volatility"] = (
    data["Monthly_Return"].rolling(12).std()
)

MIN_YEAR = int(data["Year"].min())

MAX_YEAR = int(data["Year"].max())

print("Dataset loaded successfully!")

print(f"Available years: {MIN_YEAR} - {MAX_YEAR}")

print(f"Number of monthly records: {len(data)}")


# ============================================================
# 4. DASHBOARD WIDGETS
# ============================================================

start_year = widgets.IntSlider(
    value=MIN_YEAR,
    min=MIN_YEAR,
    max=MAX_YEAR,
    step=1,
    description="Start Year:",
    continuous_update=False,
    style={"description_width": "100px"},
    layout=widgets.Layout(width="450px")
)

end_year = widgets.IntSlider(
    value=MAX_YEAR,
    min=MIN_YEAR,
    max=MAX_YEAR,
    step=1,
    description="End Year:",
    continuous_update=False,
    style={"description_width": "100px"},
    layout=widgets.Layout(width="450px")
)

exchange_rate = widgets.FloatText(
    value=85.0,
    description="USD to INR:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="250px")
)

moving_average = widgets.Dropdown(
    options=[
        ("No Moving Average", 0),
        ("3-Month Moving Average", 3),
        ("6-Month Moving Average", 6),
        ("12-Month Moving Average", 12),
        ("24-Month Moving Average", 24)
    ],
    value=12,
    description="Moving Avg:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="350px")
)


# ============================================================
# 5. GRAPH SELECTION CHECKBOXES
# ============================================================

select_all_check = widgets.Checkbox(
    value=True,
    description="Select All Graphs",
    indent=False
)

graph1_check = widgets.Checkbox(
    value=True,
    description="1. Line Graph - Gold Price Trend",
    indent=False
)

graph2_check = widgets.Checkbox(
    value=True,
    description="2. Bar Graph - Average Gold Price",
    indent=False
)

graph3_check = widgets.Checkbox(
    value=True,
    description="3. Scatter Graph - Gold Price Distribution",
    indent=False
)

graph4_check = widgets.Checkbox(
    value=True,
    description="4. Heatmap Graph - Correlation Analysis",
    indent=False
)

individual_graph_checks = [
    graph1_check,
    graph2_check,
    graph3_check,
    graph4_check
]


# Output area

output = widgets.Output()

_checkbox_guard = {"active": False}


# ============================================================
# 6. SELECT ALL CHECKBOX FUNCTION
# ============================================================

def select_all_graphs(change):

    if _checkbox_guard["active"]:
        return

    _checkbox_guard["active"] = True

    try:
        for checkbox in individual_graph_checks:
            checkbox.value = change["new"]

    finally:
        _checkbox_guard["active"] = False

    update_dashboard()


def update_select_all(change):

    if _checkbox_guard["active"]:
        return

    all_selected = all(
        checkbox.value
        for checkbox in individual_graph_checks
    )

    _checkbox_guard["active"] = True

    try:
        select_all_check.value = all_selected

    finally:
        _checkbox_guard["active"] = False

    update_dashboard()


select_all_check.observe(
    select_all_graphs,
    names="value"
)

for checkbox in individual_graph_checks:

    checkbox.observe(
        update_select_all,
        names="value"
    )


# ============================================================
# 7. HELPER FUNCTIONS
# ============================================================

def convert_to_inr_per_10g(price_usd_per_ounce, rate):

    """
    Convert USD per troy ounce into approximate
    INR per 10 grams.

    One troy ounce = 31.1034768 grams.
    """

    return (
        price_usd_per_ounce
        * rate
        * 10 / 31.1034768
    )


def format_inr(value):

    if pd.isna(value):
        return "N/A"

    return f"₹{value:,.0f}"


# ============================================================
# 8. DASHBOARD UPDATE FUNCTION
# ============================================================

def update_dashboard(change=None):

    with output:

        clear_output(wait=True)

        # Validate year selection

        first_year = start_year.value

        last_year = end_year.value

        if first_year > last_year:

            print("Error: Start Year cannot be after End Year.")

            return

        # Validate exchange rate

        rate = exchange_rate.value

        if not np.isfinite(rate) or rate <= 0:

            print("Please enter a valid positive USD-to-INR rate.")

            return

        # Select requested date range

        filtered = data[
            (data["Year"] >= first_year) &
            (data["Year"] <= last_year)
        ].copy()

        if filtered.empty:

            print("No data is available for this year range.")

            return

        # Convert USD to INR per 10 grams

        filtered["Gold_INR_per_10g"] = (
            convert_to_inr_per_10g(
                filtered["Gold_USD_per_Ounce"],
                rate
            )
        )

        # Calculate selected moving average

        ma_period = moving_average.value

        if ma_period > 0:

            filtered["Selected_MA"] = (
                filtered["Gold_INR_per_10g"]
                .rolling(ma_period)
                .mean()
            )

        # ----------------------------------------------------
        # SELECTED GRAPHS
        # ----------------------------------------------------

        selected = []

        if graph1_check.value:
            selected.append("Line Graph")

        if graph2_check.value:
            selected.append("Bar Graph")

        if graph3_check.value:
            selected.append("Scatter Graph")

        if graph4_check.value:
            selected.append("Heatmap Graph")

        # ----------------------------------------------------
        # KPI SUMMARY
        # ----------------------------------------------------

        latest_price = filtered[
            "Gold_INR_per_10g"
        ].iloc[-1]

        average_price = filtered[
            "Gold_INR_per_10g"
        ].mean()

        maximum_price = filtered[
            "Gold_INR_per_10g"
        ].max()

        first_price = filtered[
            "Gold_INR_per_10g"
        ].iloc[0]

        period_change = (
            (latest_price - first_price)
            / first_price * 100
            if first_price != 0 else np.nan
        )

        print("=" * 65)

        print("                 GOLD INSIGHT INDIA")

        print("=" * 65)

        print(f"Selected period : {first_year} - {last_year}")

        print(
            f"Latest price    : {format_inr(latest_price)} per 10 grams"
        )

        print(
            f"Average price   : {format_inr(average_price)} per 10 grams"
        )

        print(
            f"Maximum price   : {format_inr(maximum_price)} per 10 grams"
        )

        if pd.notna(period_change):

            print(f"Period change   : {period_change:+.2f}%")

        print("-" * 65)

        if not selected:

            print("Please select at least one graph checkbox.")

            return

        # ----------------------------------------------------
        # PREPARE SUBPLOTS
        # ----------------------------------------------------

        number_of_graphs = len(selected)

        number_of_rows = ceil(number_of_graphs / 2)

        fig, axes = plt.subplots(
            number_of_rows,
            2,
            figsize=(16, 5.5 * number_of_rows)
        )

        axes = np.array(axes, dtype=object).reshape(-1)

        # ----------------------------------------------------
        # GRAPH 1: LINE GRAPH
        # ----------------------------------------------------

        if "Line Graph" in selected:

            ax = axes[selected.index("Line Graph")]

            x = filtered["Date"]

            y = filtered["Gold_INR_per_10g"]

            ax.plot(
                x,
                y,
                linewidth=2,
                label="Gold Price"
            )

            # Moving Average

            if ma_period > 0:

                ax.plot(
                    x,
                    filtered["Selected_MA"],
                    linewidth=2,
                    linestyle="--",
                    label=f"{ma_period}-Month Moving Average"
                )

            ax.set_title(
                "Gold Price Trend in India",
                fontsize=13,
                fontweight="bold"
            )

            ax.set_xlabel("Year")

            ax.set_ylabel("INR per 10 grams")

            ax.tick_params(axis="x", rotation=30)

            ax.grid(True, alpha=0.3)

            ax.legend()

        # ----------------------------------------------------
        # GRAPH 2: BAR GRAPH
        # ----------------------------------------------------

        if "Bar Graph" in selected:

            ax = axes[selected.index("Bar Graph")]

            monthly_avg = (
                filtered.groupby("Month")[
                    "Gold_INR_per_10g"
                ]
                .mean()
                .reindex(range(1, 13))
            )

            month_labels = [
                "Jan", "Feb", "Mar", "Apr",
                "May", "Jun", "Jul", "Aug",
                "Sep", "Oct", "Nov", "Dec"
            ]

            ax.bar(
                month_labels,
                monthly_avg.values
            )

            ax.set_title(
                "Average Gold Price by Month",
                fontsize=13,
                fontweight="bold"
            )

            ax.set_xlabel("Month")

            ax.set_ylabel("INR per 10 grams")

            ax.grid(True, axis="y", alpha=0.3)

        # ----------------------------------------------------
        # GRAPH 3: SCATTER GRAPH
        # ----------------------------------------------------

        if "Scatter Graph" in selected:

            ax = axes[selected.index("Scatter Graph")]

            ax.scatter(
                filtered["Date"],
                filtered["Gold_INR_per_10g"],
                alpha=0.7,
                s=35
            )

            ax.set_title(
                "Gold Price Distribution Over Time",
                fontsize=13,
                fontweight="bold"
            )

            ax.set_xlabel("Year")

            ax.set_ylabel("INR per 10 grams")

            ax.tick_params(axis="x", rotation=30)

            ax.grid(True, alpha=0.3)

        # ----------------------------------------------------
        # GRAPH 4: HEATMAP GRAPH
        # ----------------------------------------------------

        if "Heatmap Graph" in selected:

            ax = axes[selected.index("Heatmap Graph")]

            corr_data = pd.DataFrame({

                "Gold Price": filtered[
                    "Gold_INR_per_10g"
                ],

                "Monthly Return (%)": (
                    filtered["Gold_INR_per_10g"]
                    .pct_change() * 100
                ),

                "3-Month Average": (
                    filtered["Gold_INR_per_10g"]
                    .rolling(3).mean()
                ),

                "12-Month Average": (
                    filtered["Gold_INR_per_10g"]
                    .rolling(12).mean()
                ),

                "Rolling Volatility": (
                    filtered["Gold_INR_per_10g"]
                    .pct_change()
                    .rolling(12).std() * 100
                )

            })

            # Remove empty and constant columns

            corr_data = corr_data.dropna(
                axis=1,
                how="all"
            )

            corr_data = corr_data.loc[
                :,
                corr_data.nunique(dropna=True) > 1
            ]

            if (
                len(corr_data) >= 3 and
                corr_data.shape[1] >= 2
            ):

                correlation = corr_data.corr()

                sns.heatmap(
                    correlation,
                    annot=True,
                    cmap="coolwarm",
                    vmin=-1,
                    vmax=1,
                    center=0,
                    fmt=".2f",
                    linewidths=0.5,
                    ax=ax
                )

            else:

                ax.text(
                    0.5,
                    0.5,
                    "Insufficient data for correlation heatmap.\n"
                    "Try selecting a longer time period.",
                    ha="center",
                    va="center",
                    transform=ax.transAxes
                )

                ax.set_axis_off()

            ax.set_title(
                "Correlation Between Gold Price Features",
                fontsize=13,
                fontweight="bold"
            )

        # ----------------------------------------------------
        # HIDE UNUSED SUBPLOTS
        # ----------------------------------------------------

        for i in range(number_of_graphs, len(axes)):

            fig.delaxes(axes[i])

        plt.tight_layout()

        plt.show()

        # ----------------------------------------------------
        # INTERPRETATION GUIDE
        # ----------------------------------------------------

        print("\nINTERPRETATION GUIDE")

        print("1. Line Graph: Shows the gold price trend over time.")

        print("2. Bar Graph: Compares average gold prices across months.")

        print("3. Scatter Graph: Shows the distribution of gold prices over time.")

        print("4. Heatmap Graph: Shows correlations between gold price features.")

        print("\nNote: INR values are approximate conversions, not actual")

        print("historical Indian retail gold prices.")

        print("The exchange rate is editable and applied uniformly.")

        print("Moving Average is displayed on the Line Graph.")


# ============================================================
# 9. CONNECT WIDGETS TO DASHBOARD
# ============================================================

other_controls = [

    start_year,

    end_year,

    exchange_rate,

    moving_average

]

for control in other_controls:

    control.observe(
        update_dashboard,
        names="value"
    )


# ============================================================
# 10. DISPLAY DASHBOARD
# ============================================================

print("\nConfigure your dashboard below.")

print("Select individual checkboxes to show or hide graphs.\n")


# Timeline controls

year_controls = widgets.VBox([

    start_year,

    end_year

])


# Currency control

price_controls = widgets.HBox([

    exchange_rate

])


# Moving average control

chart_controls = widgets.HBox([

    moving_average

])


# Graph selection

graph_selection = widgets.VBox([

    widgets.HTML("<h3>Choose Graphs to Display</h3>"),

    select_all_check,

    widgets.HBox([

        graph1_check,

        graph2_check

    ]),

    widgets.HBox([

        graph3_check,

        graph4_check

    ])

])


# Main dashboard

dashboard = widgets.VBox([

    widgets.HTML("<h2>GOLD INSIGHT INDIA</h2>"),

    widgets.HTML(
        "<b>Interactive Gold Price Analysis Dashboard</b>"
    ),

    year_controls,

    price_controls,

    chart_controls,

    graph_selection,

    output

])


# Display dashboard

display(dashboard)


# Initial dashboard display

update_dashboard()

# ============================================================
# END OF GOLD INSIGHT INDIA DASHBOARD
# ============================================================

Loading gold price dataset...
Dataset loaded successfully!
Available years: 1833 - 2026
Number of monthly records: 2326

Configure your dashboard below.
Select individual checkboxes to show or hide graphs.

